In [1]:
from IPython.display import display_svg
from typing import List, Tuple
import random
import torch
import numpy as np
import pandas as pd
from itertools import combinations_with_replacement
from typing import Callable, Any
import pyrtl
from pyrtl.rtllib.libutils import twos_comp_repr, rev_twos_comp_repr
from pyrtl import (
    WireVector, 
    Const, 
    Input,
    Output, 
    Register, 
    Simulation, 
    SimulationTrace, 
    reset_working_block
)

from kai.src.utils import custom_render_trace, basic_circuit_analysis
from kai.src.bfloat16 import BF16

In [7]:
E_BITS  = 8
M_BITS  = 7
BIAS    = 2**(E_BITS - 1) - 1
MSB     = E_BITS + M_BITS

In [14]:
max_exp = 2**E_BITS - 2
max_exp_unbiased = max_exp - BIAS

print(f"{max_exp=}, {format(max_exp, '08b')}")
print(f"{max_exp_unbiased=}, {format(max_exp_unbiased, '08b')}")

max_exp=254, 11111110
max_exp_unbiased=127, 01111111


In [21]:
bin(2**16-1), len('1111111111111111')

('0b1111111111111111', 16)

In [ ]:
def float_to_fixed(num):
    # Convert float to bfloat16 binary representation using PyTorch
    bf16_num = torch.tensor(num, dtype=torch.bfloat16).view(torch.uint16)
    bf16_binary = format(bf16_num.item(), '016b')
    
    # Convert binary bfloat16 to fixed point using previous function
    fixed_point = bfloat16_to_fixed(bf16_binary)
    
    return {
        'original_float': num,
        'bfloat16_binary': bf16_binary,
        'fixed_point': fixed_point,
        'fixed_point_binary': format(fixed_point & ((1 << 32) - 1), '032b')
    }

def bfloat16_to_fixed(bfloat16_bin):
    # Extract sign, exponent and mantissa from bfloat16 binary string
    sign = int(bfloat16_bin[0])
    exponent = int(bfloat16_bin[1:9], 2) 
    mantissa = int(bfloat16_bin[9:], 2)

    # Handle special cases
    if exponent == 0:
        if mantissa == 0:
            return 0  # Zero
        else:
            # Denormalized numbers
            exponent = -126
            mantissa = mantissa
    elif exponent == 255:
        if mantissa == 0:
            return float('inf') if sign == 0 else float('-inf')  # Infinity
        else:
            return float('nan')  # NaN
    else:
        # Normalized numbers
        exponent = exponent - 127  # Remove bias
        mantissa = mantissa | (1 << 7)  # Add implicit 1

    # Convert to Q16.16 fixed point
    # First calculate the floating point value
    float_val = ((-1) ** sign) * mantissa * (2 ** (exponent - 7))
    
    # Convert to Q16.16 by multiplying by 2^16
    fixed_point = int(float_val * (2 ** 16))
    
    # Clamp to Q16.16 range
    max_val = (2 ** 31) - 1
    min_val = -(2 ** 31)
    fixed_point = min(max_val, max(min_val, fixed_point))
    
    return fixed_point

def repr_fixed_point(fixed_point: int, bits: int = 32, signed: bool = True) -> str:

# Example usage
if __name__ == "__main__":
    test_numbers = [3.14, 1.0, -2.5, 0.5, 10.75]
    
    for num in test_numbers:
        result = float_to_fixed(num)
        print(f"\nInput number: {result['original_float']}")
        print(f"BFloat16 binary: {result['bfloat16_binary']}")
        print(f"Q16.16 fixed point value: {result['fixed_point']}")
        print(f"Q16.16 binary: {result['fixed_point_binary']}")
        
        # Print as decimal for verification
        fixed_point_decimal = result['fixed_point'] / (2**16)
        print(f"Fixed point as decimal: {fixed_point_decimal}")


Input number: 3.14
BFloat16 binary: 0100000001001001
Q16.16 fixed point value: 205824
Q16.16 binary: 00000000000000110010010000000000
Fixed point as decimal: 3.140625

Input number: 1.0
BFloat16 binary: 0011111110000000
Q16.16 fixed point value: 65536
Q16.16 binary: 00000000000000010000000000000000
Fixed point as decimal: 1.0

Input number: -2.5
BFloat16 binary: 1100000000100000
Q16.16 fixed point value: -163840
Q16.16 binary: 11111111111111011000000000000000
Fixed point as decimal: -2.5

Input number: 0.5
BFloat16 binary: 0011111100000000
Q16.16 fixed point value: 32768
Q16.16 binary: 00000000000000001000000000000000
Fixed point as decimal: 0.5

Input number: 10.75
BFloat16 binary: 0100000100101100
Q16.16 fixed point value: 704512
Q16.16 binary: 00000000000010101100000000000000
Fixed point as decimal: 10.75


In [27]:
num = 3.14

bf16_num = torch.tensor(num, dtype=torch.bfloat16).view(torch.uint16)
format(bf16_num, '016b')

'0100000001001001'